# SERL Calibration Pipeline (Deterministic)

This notebook is intentionally self-contained and designed for strict top-to-bottom execution.

It calibrates ABM parameters against SERL marginals across multiple `seg3_var` dimensions using coordinate descent (one parameter at a time).


## Workflow

1. Load SERL targets and WF cohort.
2. Map WF features to SERL-like segment labels.
3. Run ABM with segmentation aggregation enabled.
4. Score ABM vs SERL across multiple marginals.
5. Tune parameters one-at-a-time (coordinate descent).
6. Save run artefacts (`trace`, `final_params`, tested configs).


In [4]:
from __future__ import annotations

import sys
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import geopandas as gpd
import matplotlib.pyplot as plt

# Always import local repo modules (avoid stale installed package behavior)
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import importlib
import household_energy.model as _hem
import household_energy.climate as _hec
importlib.reload(_hem)
importlib.reload(_hec)

EnergyModel = _hem.EnergyModel
ClimateField = _hec.ClimateField


In [5]:
# ---- User configuration ----
TARGET_YEAR = 2023
LOCAL_TZ = 'Europe/London'
FAST_DAYS = 14                 # window length per calibration slice
MAX_HOMES = 1500               # runtime control
SEED = 7

# Restrict if needed, e.g. ['num_occupants','floor_area_m2']
SEG3_VARS_OVERRIDE = None

# Objective weights across strict components (lower is better)
DEFAULT_WEIGHTS = {
    'electric_diurnal_shape_rmse': 1.8,
    'gas_diurnal_shape_rmse': 1.8,
    'electric_monthly_level_rmse': 1.2,
    'gas_monthly_level_rmse': 1.2,
    'electric_hourly_level_ratio_abslog': 2.0,
    'gas_hourly_level_ratio_abslog': 2.0,
    'electric_peak_ratio_abs': 2.0,
    'gas_peak_ratio_abs': 2.0,
    'electric_shape_corr_penalty': 1.5,
    'gas_shape_corr_penalty': 1.5,
}

# Stage-specific search spaces and weighting emphasis.
STAGE_DEFS = [
    {
        'name': 'S1_summer_baseline',
        'description': 'Summer baseline fit (Jul-Sep robust low-quantile, by seg3)',
        'params': [
            ('model.use_separate_fuel_baseline_anchors', [True]),
            ('model.baseline_anchor_elec_kwh_per_hour', [0.14, 0.18, 0.22, 0.26]),
            ('model.baseline_anchor_gas_kwh_per_hour', [0.00, 0.01, 0.02, 0.03]),
            ('model.energy_per_person_home', [0.03, 0.04, 0.05]),
            ('model.energy_per_person_away', [0.002, 0.004, 0.006]),
        ],
        'weights': {
            **DEFAULT_WEIGHTS,
            'electric_monthly_level_rmse': 2.6,
            'gas_monthly_level_rmse': 3.0,
            'electric_hourly_level_ratio_abslog': 2.6,
            'gas_hourly_level_ratio_abslog': 3.0,
            'electric_peak_ratio_abs': 0.7,
            'gas_peak_ratio_abs': 0.7,
        },
        'month_filter': [7, 8, 9],
        'monthly_hdd_max': 1.5,
    },
    {
        'name': 'S2_summer_peaks',
        'description': 'Summer AM/PM peak structure fit (separate morning/evening)',
        'params': [
            ('model.heating_peak_morning_mult', [0.85, 1.00, 1.15, 1.30]),
            ('model.heating_peak_evening_mult', [0.90, 1.05, 1.20, 1.35]),
            ('model.dhw_peak_morning_mult', [0.90, 1.05, 1.20, 1.35]),
            ('model.dhw_peak_evening_mult', [0.90, 1.05, 1.20, 1.35]),
            ('model.gas_spike_share', [0.00, 0.08, 0.16, 0.24]),
            ('model.dhw_daily_kwh_per_home', [0.0, 0.2, 0.4, 0.6]),
            ('model.dhw_daily_kwh_per_person', [0.0, 0.10, 0.20, 0.30]),
            ('model.dhw_away_mult', [0.2, 0.35, 0.5]),
            ('schedules.wfh_share', [0.0, 0.2, 0.4]),
            ('schedules.jitter_hours', [0, 1, 2]),
        ],
        'weights': {
            **DEFAULT_WEIGHTS,
            'electric_diurnal_shape_rmse': 2.4,
            'gas_diurnal_shape_rmse': 2.8,
            'electric_peak_ratio_abs': 2.8,
            'gas_peak_ratio_abs': 3.4,
            'electric_shape_corr_penalty': 2.2,
            'gas_shape_corr_penalty': 2.6,
            'gas_monthly_level_rmse': 2.2,
        },
        'month_filter': [7, 8, 9],
        'monthly_hdd_max': 1.5,
        'guardrails': {
            'months': [7, 8, 9],
            'ratio_band': [0.9, 1.1],
            'fuel_weights': {'gas': 30.0, 'electric': 10.0},
        },
    },
    {
        'name': 'W1_winter_uplift',
        'description': 'Winter uplift vs temperature slope + winter AM/PM peak modifiers',
        'params': [
            ('model.heating_setpoint_C', [18.0, 18.5, 19.0]),
            ('model.setpoint_setback_C', [1.0, 2.0, 3.0]),
            ('model.heating_slope_kWh_per_deg', [0.04, 0.05, 0.06]),
            ('model.loss_to_duty_k', [2.0, 3.0, 4.0]),
            ('model.max_heat_kwh_per_hour', [18.0, 24.0, 30.0]),
            ('model.heating_occupancy_away_mult', [0.35, 0.50, 0.65]),
            ('model.heating_winter_morning_mult', [0.90, 1.05, 1.20, 1.35]),
            ('model.heating_winter_evening_mult', [0.95, 1.10, 1.25, 1.40]),
        ],
        'weights': {
            **DEFAULT_WEIGHTS,
            'gas_monthly_level_rmse': 3.2,
            'gas_diurnal_shape_rmse': 2.3,
            'gas_peak_ratio_abs': 2.8,
            'gas_shape_corr_penalty': 2.2,
            'electric_monthly_level_rmse': 1.8,
        },
        'month_filter': [11, 12, 1, 2, 3],
        'monthly_hdd_min': 3.0,
    },
    {
        'name': 'W2_joint_refine',
        'description': 'Short joint refinement',
        'params': [
            ('model.baseline_anchor_elec_kwh_per_hour', [-0.02, 0.0, 0.02]),
            ('model.baseline_anchor_gas_kwh_per_hour', [-0.01, 0.0, 0.01]),
            ('model.heating_slope_kWh_per_deg', [-0.006, 0.0, 0.006]),
            ('model.gas_spike_share', [-0.04, 0.0, 0.04]),
            ('model.heating_peak_morning_mult', [-0.10, 0.0, 0.10]),
            ('model.heating_peak_evening_mult', [-0.10, 0.0, 0.10]),
            ('model.heating_winter_morning_mult', [-0.10, 0.0, 0.10]),
            ('model.heating_winter_evening_mult', [-0.10, 0.0, 0.10]),
        ],
        'weights': dict(DEFAULT_WEIGHTS),
        'relative': True,
    },
]

In [ ]:
# ---- Paths + input checks ----
DATA_DIR = (REPO_ROOT / 'data').resolve()
TARGET_DIR = (DATA_DIR / 'serl_8963_targets').resolve()
EPC_GEOJSON = DATA_DIR / 'epc_abm_waltham_forest.geojson'
HIDP_CSV = DATA_DIR / 'wf_hidp_uprn_matches_tiered.csv'
CLIMATE_PARQUET = DATA_DIR / 'waltham_forest_2t_timeseries_2010_2026.parquet'

required = [
    TARGET_DIR / 'daily_targets.csv',
    TARGET_DIR / 'diurnal_targets_hourly_mean.csv',
    EPC_GEOJSON,
    HIDP_CSV,
    CLIMATE_PARQUET,
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required inputs:' + ''.join(missing))

OUT_DIR = (REPO_ROOT / 'results' / f"serl_calib_pipeline_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}").resolve()
CONFIGS_DIR = OUT_DIR / 'configs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

print('Output dir:', OUT_DIR)


SyntaxError: unterminated string literal (detected at line 17) (3374075153.py, line 17)

In [ ]:
@dataclass(frozen=True)
class Window:
    name: str
    start_utc: pd.Timestamp
    end_utc: pd.Timestamp


def rmse(a, b) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() == 0:
        return float('nan')
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))


def wf_make_serl_bands(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    out = gdf.copy()

    occ = pd.to_numeric(out.get('hh_n_people'), errors='coerce')
    def occ_band(x):
        if pd.isna(x):
            return 'No data'
        x = int(x)
        if x >= 6:
            return '>=6'
        return str(max(1, x))
    out['serl_num_occupants'] = occ.apply(occ_band)

    area = None
    for k in ['floor_area_m2', 'totalFloorArea', 'totalfloorarea', 'TOTAL_FLOOR_AREA', 'total_floor_area']:
        if k in out.columns:
            area = pd.to_numeric(out[k], errors='coerce')
            break
    if area is None:
        area = pd.Series(np.nan, index=out.index)

    def area_band(a):
        if pd.isna(a):
            return 'No data'
        a = float(a)
        if a <= 50:
            return '50 or less'
        if a <= 100:
            return '51 to 100'
        if a <= 150:
            return '101 to 150'
        if a <= 200:
            return '151 to 200'
        return 'Over 200'
    out['serl_floor_area_m2'] = area.apply(area_band)

    epc = None
    for k in ['currentEnergyRating', 'current_energy_rating', 'currentenergyrating', 'CURRENT_ENERGY_RATING']:
        if k in out.columns:
            epc = out[k]
            break
    if epc is None:
        out['serl_currentEnergyRating'] = 'No data'
    else:
        out['serl_currentEnergyRating'] = epc.astype(str).str.strip().replace({'': 'No data', 'nan': 'No data'})

    ptype = out.get('property_type')
    if ptype is None:
        out['serl_building_type'] = 'No data'
    else:
        t = ptype.astype(str).str.lower()
        out['serl_building_type'] = np.select(
            [
                t.str.contains('detached', na=False) & ~t.str.contains('semi', na=False),
                t.str.contains('semi', na=False),
                t.str.contains('terraced', na=False),
                t.str.contains('flat', na=False) | t.str.contains('flats', na=False),
            ],
            ['Detached', 'Semi-detached', 'Terraced', 'Flat'],
            default='Other/Unknown',
        )

    imd = None
    imd_src = None
    for k in ['imd_quintile', 'IMD_quintile', 'imd_decile', 'IMD_decile']:
        if k in out.columns:
            imd = pd.to_numeric(out[k], errors='coerce')
            imd_src = k
            break
    if imd is None:
        out['serl_IMD_quintile'] = 'No data'
    else:
        if 'decile' in str(imd_src).lower():
            q = np.ceil(imd / 2.0)
        else:
            q = imd
        q = q.where(q.between(1, 5), np.nan)
        out['serl_IMD_quintile'] = q.fillna('No data').astype(str)

    return out


def set_deep(d: dict, path: list[str], value):
    cur = d
    for key in path[:-1]:
        if key not in cur or not isinstance(cur[key], dict):
            cur[key] = {}
        cur = cur[key]
    cur[path[-1]] = value


def make_override_yaml(*, name: str, params: dict) -> dict:
    payload = {
        'meta': {'name': name, 'date': pd.Timestamp.utcnow().date().isoformat(), 'notes': 'serl calibration pipeline'},
        'model': {},
        'schedules': {},
        'calibration': {},
        'serl_profiles': {},
    }
    for key, value in params.items():
        if '.' not in key:
            continue
        top, rest = key.split('.', 1)
        if top not in payload:
            continue
        set_deep(payload[top], rest.split('.'), value)
    return payload


def abm_profiles_from_seg_ts(seg_ts: pd.DataFrame, *, local_tz: str):
    df = seg_ts.copy()
    df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True)
    loc = df['timestamp_utc'].dt.tz_convert(local_tz)
    df['local_hour'] = loc.dt.hour
    df['local_date'] = loc.dt.floor('D')
    df['local_month'] = loc.dt.month

    df['elec_kwh_per_home'] = df['electric_kwh'] / df['n_homes'].replace({0: np.nan})
    df['gas_kwh_per_home'] = df['gas_kwh'] / df['n_homes'].replace({0: np.nan})

    hourly = (df.groupby(['segmentation', 'value', 'local_hour'], as_index=False)
        .agg(elec_kwh=('elec_kwh_per_home', 'mean'), gas_kwh=('gas_kwh_per_home', 'mean'))
        .sort_values(['segmentation', 'value', 'local_hour']))

    daily = (df.groupby(['segmentation', 'value', 'local_date', 'local_month'], as_index=False)
        .agg(electric_kwh_day=('electric_kwh', 'sum'), gas_kwh_day=('gas_kwh', 'sum'), n=('n_homes', 'min')))
    daily['elec_kwh_per_home_day'] = daily['electric_kwh_day'] / daily['n'].replace({0: np.nan})
    daily['gas_kwh_per_home_day'] = daily['gas_kwh_day'] / daily['n'].replace({0: np.nan})

    monthly = (daily.groupby(['segmentation', 'value', 'local_month'], as_index=False)
        .agg(elec_kwh_per_home_day=('elec_kwh_per_home_day', 'mean'),
             gas_kwh_per_home_day=('gas_kwh_per_home_day', 'mean'))
        .rename(columns={'local_month': 'month'})
        .sort_values(['segmentation', 'value', 'month']))

    return hourly, monthly


def score_against_serl_marginals(
    *, abm_hourly, abm_monthly, hourly_targets, daily_targets, target_year, seg3_vars,
    month_filter: list[int] | None = None,
    monthly_hdd_min: float | None = None,
    monthly_hdd_max: float | None = None,
):
    rows = []
    eps = 1e-9
    month_filter_set = set(int(m) for m in month_filter) if month_filter else None

    for seg3 in seg3_vars:
        hr_t = hourly_targets[(hourly_targets['year'].astype(int) == int(target_year))
            & (hourly_targets['weekday_weekend'].astype(str) == 'both')
            & (hourly_targets['heating_fuel'].astype(str) == 'All')
            & (hourly_targets['has_pv'].astype(str).isin(['No', 'All']))
            & (hourly_targets['seg3_var'].astype(str) == str(seg3))].copy()

        mo_t = daily_targets[(daily_targets['period_type'].astype(str) == 'monthly')
            & (daily_targets['year'].astype(int) == int(target_year))
            & (daily_targets['weekday_weekend'].astype(str) == 'both')
            & (daily_targets['heating_fuel'].astype(str) == 'All')
            & (daily_targets['has_pv'].astype(str).isin(['No', 'All']))
            & (daily_targets['seg3_var'].astype(str) == str(seg3))].copy()

        ahr = abm_hourly[abm_hourly['segmentation'] == seg3].copy()
        amo = abm_monthly[abm_monthly['segmentation'] == seg3].copy()

        if month_filter_set is not None:
            mo_t = mo_t[mo_t['month'].astype('Int64').isin(month_filter_set)].copy()
            amo = amo[amo['month'].astype('Int64').isin(month_filter_set)].copy()

        if monthly_hdd_min is not None and 'mean_hdd' in mo_t.columns:
            mo_t = mo_t[pd.to_numeric(mo_t['mean_hdd'], errors='coerce') >= float(monthly_hdd_min)].copy()
        if monthly_hdd_max is not None and 'mean_hdd' in mo_t.columns:
            mo_t = mo_t[pd.to_numeric(mo_t['mean_hdd'], errors='coerce') <= float(monthly_hdd_max)].copy()

        for q, col_abm, prefix in [
            ('Electricity imports', 'elec_kwh', 'electric'),
            ('Gas', 'gas_kwh', 'gas'),
        ]:
            st = hr_t[hr_t['quantity'].astype(str) == q].copy()
            if st.empty:
                continue
            merged = ahr.merge(
                st[['seg3_value', 'hour', 'mean_kwh']].rename(columns={'seg3_value': 'value', 'hour': 'local_hour'}),
                on=['value', 'local_hour'], how='inner')
            if merged.empty:
                continue

            per_value = []
            for _, g in merged.groupby('value', observed=True):
                a = g[col_abm].to_numpy(dtype=float)
                s = g['mean_kwh'].to_numpy(dtype=float)
                if len(a) == 0 or np.nansum(a) <= 0 or np.nansum(s) <= 0:
                    continue

                a_shape = a / max(np.nansum(a), eps)
                s_shape = s / max(np.nansum(s), eps)
                shape_rmse = rmse(a_shape, s_shape)

                mean_a = float(np.nanmean(a)); mean_s = float(np.nanmean(s))
                level_abslog = float(abs(np.log(max(mean_a, eps) / max(mean_s, eps))))

                peak_ratio_a = float(np.nanmax(a) / max(mean_a, eps))
                peak_ratio_s = float(np.nanmax(s) / max(mean_s, eps))
                peak_abs = float(abs(peak_ratio_a - peak_ratio_s))

                corr = float(np.corrcoef(a_shape, s_shape)[0, 1]) if len(a_shape) > 1 else np.nan
                if not np.isfinite(corr):
                    corr = -1.0
                corr_penalty = float(max(0.0, 1.0 - corr))

                per_value.append((shape_rmse, level_abslog, peak_abs, corr_penalty))

            if per_value:
                arr = np.asarray(per_value, dtype=float)
                rows.append({'seg3_var': seg3, 'component': f'{prefix}_diurnal_shape_rmse', 'value': float(np.nanmean(arr[:, 0]))})
                rows.append({'seg3_var': seg3, 'component': f'{prefix}_hourly_level_ratio_abslog', 'value': float(np.nanmean(arr[:, 1]))})
                rows.append({'seg3_var': seg3, 'component': f'{prefix}_peak_ratio_abs', 'value': float(np.nanmean(arr[:, 2]))})
                rows.append({'seg3_var': seg3, 'component': f'{prefix}_shape_corr_penalty', 'value': float(np.nanmean(arr[:, 3]))})

        for q, col_abm, out_name in [
            ('Electricity imports', 'elec_kwh_per_home_day', 'electric_monthly_level_rmse'),
            ('Gas', 'gas_kwh_per_home_day', 'gas_monthly_level_rmse'),
        ]:
            st = mo_t[mo_t['quantity'].astype(str) == q].copy()
            if st.empty:
                continue
            merged = amo.merge(st[['seg3_value', 'month', 'mean']].rename(columns={'seg3_value': 'value'}), on=['value', 'month'], how='inner')
            if merged.empty:
                continue
            byv = (merged.groupby('value', as_index=False)
                .agg(r=(col_abm, lambda s: rmse(s.to_numpy(), merged.loc[s.index, 'mean'].to_numpy()))))
            rows.append({'seg3_var': seg3, 'component': out_name, 'value': float(byv['r'].mean())})

    return pd.DataFrame(rows)


def scalar_objective(marg_scores: pd.DataFrame, weights: dict) -> float:
    if marg_scores.empty:
        return float('nan')
    by = marg_scores.groupby('component', as_index=False).agg(v=('value', 'mean'))
    return float(sum(weights.get(r['component'], 0.0) * float(r['v']) for _, r in by.iterrows()))


def monthly_ratio_guardrail_penalty(
    *, abm_monthly: pd.DataFrame, daily_targets: pd.DataFrame, target_year: int, seg3_vars: list[str],
    months: list[int], ratio_band: tuple[float, float], fuel_weights: dict[str, float],
) -> tuple[float, pd.DataFrame]:
    lo, hi = float(ratio_band[0]), float(ratio_band[1])
    months_set = {int(m) for m in months}
    rows = []

    base = daily_targets[(daily_targets['period_type'].astype(str) == 'monthly')
        & (daily_targets['year'].astype(int) == int(target_year))
        & (daily_targets['weekday_weekend'].astype(str) == 'both')
        & (daily_targets['heating_fuel'].astype(str) == 'All')
        & (daily_targets['has_pv'].astype(str).isin(['No', 'All']))
        & (daily_targets['seg3_var'].astype(str).isin(seg3_vars))
        & (daily_targets['month'].astype('Int64').isin(months_set))].copy()

    if base.empty:
        return 0.0, pd.DataFrame(columns=['fuel', 'month', 'weighted_ratio', 'penalty'])

    for fuel, q, abm_col in [
        ('electric', 'Electricity imports', 'elec_kwh_per_home_day'),
        ('gas', 'Gas', 'gas_kwh_per_home_day'),
    ]:
        w = float(fuel_weights.get(fuel, 0.0))
        if w <= 0:
            continue

        s = base[base['quantity'].astype(str) == q][['seg3_var', 'seg3_value', 'month', 'mean', 'n_rounded']].copy()
        m = abm_monthly[abm_monthly['segmentation'].astype(str).isin(seg3_vars)][['segmentation', 'value', 'month', abm_col]].copy()
        z = m.merge(
            s,
            left_on=['segmentation', 'value', 'month'],
            right_on=['seg3_var', 'seg3_value', 'month'],
            how='inner',
        )
        if z.empty:
            continue

        z['ratio'] = z[abm_col] / z['mean'].replace({0: np.nan})
        z['nw'] = pd.to_numeric(z['n_rounded'], errors='coerce').fillna(1.0)

        for month, gm in z.groupby('month', observed=True):
            gm = gm[np.isfinite(gm['ratio']) & (gm['ratio'] > 0)]
            if gm.empty:
                continue
            ratio = float(np.average(gm['ratio'], weights=gm['nw']))
            if ratio < lo:
                dev = abs(np.log(max(ratio, 1e-9) / lo))
            elif ratio > hi:
                dev = abs(np.log(ratio / hi))
            else:
                dev = 0.0
            pen = w * dev
            rows.append({'fuel': fuel, 'month': int(month), 'weighted_ratio': ratio, 'penalty': pen})

    if not rows:
        return 0.0, pd.DataFrame(columns=['fuel', 'month', 'weighted_ratio', 'penalty'])
    out = pd.DataFrame(rows)
    return float(out['penalty'].sum()), out



def estimate_initial_baselines_from_serl(*, hourly_targets, daily_targets, target_year, seg3_vars):
    """Estimate summer baselines from robust low quantiles (Jul-Sep).

    We avoid literal minima; use quantiles to reduce outlier sensitivity.
    """
    summer_months = [7, 8, 9]
    q_low = 0.15

    monthly = daily_targets[(daily_targets['period_type'].astype(str) == 'monthly')
        & (daily_targets['year'].astype(int) == int(target_year))
        & (daily_targets['weekday_weekend'].astype(str) == 'both')
        & (daily_targets['heating_fuel'].astype(str) == 'All')
        & (daily_targets['has_pv'].astype(str).isin(['No', 'All']))
        & (daily_targets['seg3_var'].astype(str).isin(seg3_vars))
        & (daily_targets['month'].astype('Int64').isin(summer_months))].copy()

    if 'mean_hdd' in monthly.columns:
        monthly = monthly[pd.to_numeric(monthly['mean_hdd'], errors='coerce') <= 1.5].copy()

    def _weighted_low_quantile(df: pd.DataFrame, val_col: str, w_col: str, q: float) -> float:
        if df.empty:
            return float('nan')
        v = pd.to_numeric(df[val_col], errors='coerce').to_numpy(dtype=float)
        w = pd.to_numeric(df[w_col], errors='coerce').fillna(1.0).to_numpy(dtype=float)
        m = np.isfinite(v) & np.isfinite(w) & (w > 0)
        if m.sum() == 0:
            return float('nan')
        v = v[m]; w = w[m]
        order = np.argsort(v)
        v = v[order]; w = w[order]
        cw = np.cumsum(w)
        target = q * cw[-1]
        idx = int(np.searchsorted(cw, target, side='left'))
        idx = min(max(idx, 0), len(v) - 1)
        return float(v[idx])

    e_month = monthly[monthly['quantity'].astype(str) == 'Electricity imports'].copy()
    g_month = monthly[monthly['quantity'].astype(str) == 'Gas'].copy()

    elec_daily_low = _weighted_low_quantile(e_month, 'mean', 'n_rounded', q_low)
    gas_daily_low = _weighted_low_quantile(g_month, 'mean', 'n_rounded', q_low)

    # Hour-block support (off-peak night hours) from hourly targets by seg3.
    # Hourly targets are not month-resolved in current inputs, so this is a secondary signal.
    hourly = hourly_targets[(hourly_targets['year'].astype(int) == int(target_year))
        & (hourly_targets['weekday_weekend'].astype(str) == 'both')
        & (hourly_targets['heating_fuel'].astype(str) == 'All')
        & (hourly_targets['has_pv'].astype(str).isin(['No', 'All']))
        & (hourly_targets['seg3_var'].astype(str).isin(seg3_vars))
        & (hourly_targets['hour'].astype(int).isin([1, 2, 3, 4]))].copy()

    e_hour = hourly[hourly['quantity'].astype(str) == 'Electricity imports']
    g_hour = hourly[hourly['quantity'].astype(str) == 'Gas']

    elec_hour_low = _weighted_low_quantile(e_hour, 'mean_kwh', 'n_rounded', q_low)
    gas_hour_low = _weighted_low_quantile(g_hour, 'mean_kwh', 'n_rounded', q_low)

    # Blend monthly/day and hourly estimates; clamp to plausible bounds.
    elec_anchor = np.nanmean([elec_hour_low, elec_daily_low / 24.0])
    gas_anchor = np.nanmean([gas_hour_low, gas_daily_low / 24.0])

    if not np.isfinite(elec_anchor):
        elec_anchor = 0.20
    if not np.isfinite(gas_anchor):
        gas_anchor = 0.02

    elec_anchor = float(np.clip(elec_anchor, 0.05, 0.70))
    gas_anchor = float(np.clip(gas_anchor, 0.00, 0.18))

    return {
        'model.use_separate_fuel_baseline_anchors': True,
        'model.baseline_anchor_elec_kwh_per_hour': elec_anchor,
        'model.baseline_anchor_gas_kwh_per_hour': gas_anchor,
    }



def build_stage_windows(target_year: int, fast_days: int) -> dict[str, list[Window]]:
    return {
        'S1_summer_baseline': [
            Window('summer_jul', pd.Timestamp(f'{target_year}-07-10T00:00:00Z'), pd.Timestamp(f'{target_year}-07-10T00:00:00Z') + pd.Timedelta(days=fast_days)),
            Window('summer_aug', pd.Timestamp(f'{target_year}-08-10T00:00:00Z'), pd.Timestamp(f'{target_year}-08-10T00:00:00Z') + pd.Timedelta(days=fast_days)),
            Window('summer_sep', pd.Timestamp(f'{target_year}-09-05T00:00:00Z'), pd.Timestamp(f'{target_year}-09-05T00:00:00Z') + pd.Timedelta(days=fast_days)),
        ],
        'S2_summer_peaks': [
            Window('summer_jul', pd.Timestamp(f'{target_year}-07-10T00:00:00Z'), pd.Timestamp(f'{target_year}-07-10T00:00:00Z') + pd.Timedelta(days=fast_days)),
            Window('summer_aug', pd.Timestamp(f'{target_year}-08-10T00:00:00Z'), pd.Timestamp(f'{target_year}-08-10T00:00:00Z') + pd.Timedelta(days=fast_days)),
            Window('summer_sep', pd.Timestamp(f'{target_year}-09-05T00:00:00Z'), pd.Timestamp(f'{target_year}-09-05T00:00:00Z') + pd.Timedelta(days=fast_days)),
        ],
        'W1_winter_uplift': [
            Window('winter_jan', pd.Timestamp(f'{target_year}-01-15T00:00:00Z'), pd.Timestamp(f'{target_year}-01-15T00:00:00Z') + pd.Timedelta(days=fast_days)),
            Window('winter_feb', pd.Timestamp(f'{target_year}-02-15T00:00:00Z'), pd.Timestamp(f'{target_year}-02-15T00:00:00Z') + pd.Timedelta(days=fast_days)),
            Window('winter_dec', pd.Timestamp(f'{target_year}-12-05T00:00:00Z'), pd.Timestamp(f'{target_year}-12-05T00:00:00Z') + pd.Timedelta(days=fast_days)),
        ],
        'W2_joint_refine': [
            Window('winter_jan', pd.Timestamp(f'{target_year}-01-15T00:00:00Z'), pd.Timestamp(f'{target_year}-01-15T00:00:00Z') + pd.Timedelta(days=fast_days)),
            Window('summer_aug', pd.Timestamp(f'{target_year}-08-10T00:00:00Z'), pd.Timestamp(f'{target_year}-08-10T00:00:00Z') + pd.Timedelta(days=fast_days)),
        ],
    }




def regress_winter_uplift_vs_hdd(*, abm_monthly: pd.DataFrame, daily_targets: pd.DataFrame, target_year: int, seg3_vars: list[str]) -> pd.DataFrame:
    """Estimate ABM vs SERL winter uplift slopes against HDD by seg3/fuel.

    Uplift is computed above Jul-Sep baseline (mean over available summer months).
    """
    serl = daily_targets[(daily_targets['period_type'].astype(str) == 'monthly')
        & (daily_targets['year'].astype(int) == int(target_year))
        & (daily_targets['weekday_weekend'].astype(str) == 'both')
        & (daily_targets['heating_fuel'].astype(str) == 'All')
        & (daily_targets['has_pv'].astype(str).isin(['No', 'All']))
        & (daily_targets['seg3_var'].astype(str).isin(seg3_vars))
        & (daily_targets['quantity'].astype(str).isin(['Electricity imports', 'Gas']))].copy()

    if serl.empty:
        return pd.DataFrame(columns=['seg3_var', 'fuel', 'abm_slope', 'serl_slope', 'slope_gap'])

    summer = {7, 8, 9}
    winter = {11, 12, 1, 2, 3}

    rows = []
    for seg3 in seg3_vars:
        for fuel, q, abm_col in [('electric', 'Electricity imports', 'elec_kwh_per_home_day'), ('gas', 'Gas', 'gas_kwh_per_home_day')]:
            s = serl[(serl['seg3_var'].astype(str) == str(seg3)) & (serl['quantity'].astype(str) == q)].copy()
            a = abm_monthly[abm_monthly['segmentation'].astype(str) == str(seg3)].copy()
            if s.empty or a.empty:
                continue
            m = a.merge(s[['seg3_value', 'month', 'mean', 'mean_hdd']], left_on=['value', 'month'], right_on=['seg3_value', 'month'], how='inner')
            if m.empty:
                continue

            m['month_i'] = m['month'].astype(int)
            s_base = (m[m['month_i'].isin(summer)].groupby('value', as_index=False)['mean'].mean().rename(columns={'mean': 'serl_base'}))
            a_base = (m[m['month_i'].isin(summer)].groupby('value', as_index=False)[abm_col].mean().rename(columns={abm_col: 'abm_base'}))
            mm = m.merge(s_base, on='value', how='left').merge(a_base, on='value', how='left')
            mm = mm[mm['month_i'].isin(winter)].copy()
            if mm.empty:
                continue

            mm['serl_uplift'] = mm['mean'] - mm['serl_base']
            mm['abm_uplift'] = mm[abm_col] - mm['abm_base']
            mm['hdd'] = pd.to_numeric(mm['mean_hdd'], errors='coerce')
            mm = mm[np.isfinite(mm['hdd'])]
            if len(mm) < 3:
                continue

            try:
                serl_slope = float(np.polyfit(mm['hdd'].to_numpy(dtype=float), mm['serl_uplift'].to_numpy(dtype=float), 1)[0])
                abm_slope = float(np.polyfit(mm['hdd'].to_numpy(dtype=float), mm['abm_uplift'].to_numpy(dtype=float), 1)[0])
            except Exception:
                continue

            rows.append({'seg3_var': seg3, 'fuel': fuel, 'abm_slope': abm_slope, 'serl_slope': serl_slope, 'slope_gap': abm_slope - serl_slope})

    return pd.DataFrame(rows)

def run_with_segmentations(*, wf_sample, windows, override_params, configs_dir, climate_parquet, local_tz, run_label):
    cfg_path = configs_dir / f'{run_label}.yaml'
    payload = make_override_yaml(name=run_label, params=override_params)
    with cfg_path.open('w', encoding='utf-8') as fh:
        yaml.safe_dump(payload, fh, sort_keys=False)

    seg_frames = []
    cf = ClimateField(str(climate_parquet))
    for window in windows:
        i0 = cf.time_index_for(window.start_utc)
        i1 = cf.time_index_for(window.end_utc)
        start_aligned = pd.to_datetime(cf.times[i0], utc=True)
        model = EnergyModel(
            gdf=wf_sample,
            climate_parquet=str(climate_parquet),
            climate_start=start_aligned,
            local_tz=local_tz,
            collect_agent_level=False,
            config_path=str(cfg_path),
        )
        for _ in range(int(i1 - i0)):
            model.step()
        seg_frames.append(model.get_segmentation_timeseries())

    seg_ts = pd.concat(seg_frames, ignore_index=True) if seg_frames else pd.DataFrame()
    return seg_ts, cfg_path

In [ ]:
# ---- Load data + build cohort + choose seg3 vars ----
daily_targets = pd.read_csv(TARGET_DIR / 'daily_targets.csv')
hourly_targets = pd.read_csv(TARGET_DIR / 'diurnal_targets_hourly_mean.csv')

wf = gpd.read_file(EPC_GEOJSON)
hidp = pd.read_csv(HIDP_CSV, low_memory=False)
wf['UPRN'] = wf['UPRN'].astype(str).str.strip()
hidp['uprn_chr'] = hidp['uprn_chr'].astype(str).str.strip()
wf = wf.merge(hidp, how='left', left_on='UPRN', right_on='uprn_chr')
wf = wf_make_serl_bands(wf)

WF_SEG3_MAP = {
    'num_occupants': 'serl_num_occupants',
    'floor_area_m2': 'serl_floor_area_m2',
    'currentEnergyRating': 'serl_currentEnergyRating',
    'building_type': 'serl_building_type',
    'IMD_quintile': 'serl_IMD_quintile',
}

all_seg3 = sorted({v for v in hourly_targets['seg3_var'].astype(str).unique().tolist()} | {v for v in daily_targets['seg3_var'].astype(str).unique().tolist()})
all_seg3 = [v for v in all_seg3 if v not in ('none', 'nan', '<NA>')]
seg3_vars = [v for v in all_seg3 if v in WF_SEG3_MAP]
if SEG3_VARS_OVERRIDE is not None:
    seg3_vars = [v for v in seg3_vars if v in set(SEG3_VARS_OVERRIDE)]

rng = np.random.default_rng(SEED)
if len(wf) > MAX_HOMES:
    wf = wf.iloc[rng.choice(wf.index.to_numpy(), size=MAX_HOMES, replace=False)].copy()

windows = [
    Window('winter', pd.Timestamp(f'{TARGET_YEAR}-01-15T00:00:00Z'), pd.Timestamp(f'{TARGET_YEAR}-01-15T00:00:00Z') + pd.Timedelta(days=FAST_DAYS)),
    Window('summer', pd.Timestamp(f'{TARGET_YEAR}-07-15T00:00:00Z'), pd.Timestamp(f'{TARGET_YEAR}-07-15T00:00:00Z') + pd.Timedelta(days=FAST_DAYS)),
]

calib_segmentations = [
    {'name': seg3, 'attr': WF_SEG3_MAP[seg3], 'fallback_value': 'No data'}
    for seg3 in seg3_vars
]

print('WF homes in run:', len(wf))
print('seg3 vars:', seg3_vars)


In [ ]:
# ---- Staged calibration (coordinate descent) ----
STAGE_WINDOWS = build_stage_windows(TARGET_YEAR, FAST_DAYS)

base_params = {
    'calibration.segmentations': calib_segmentations,
    'model.use_separate_fuel_baseline_anchors': True,
}
base_params.update(
    estimate_initial_baselines_from_serl(
        hourly_targets=hourly_targets,
        daily_targets=daily_targets,
        target_year=TARGET_YEAR,
        seg3_vars=seg3_vars,
    )
)

current = dict(base_params)
trace_rows: list[dict] = []
stage_metrics_rows: list[dict] = []
guardrail_rows: list[dict] = []
latest_scores_by_stage: dict[str, pd.DataFrame] = {}
run_counter = 0

def _candidate_value(stage_cfg: dict, param_name: str, current_value, candidate):
    if bool(stage_cfg.get('relative', False)):
        base = float(current_value) if current_value is not None else 0.0
        v = base + float(candidate)
    else:
        v = candidate

    if param_name in {
        'model.baseline_anchor_elec_kwh_per_hour',
        'model.baseline_anchor_gas_kwh_per_hour',
        'model.heating_slope_kWh_per_deg',
        'model.dhw_daily_kwh_per_home',
        'model.dhw_daily_kwh_per_person',
    }:
        v = max(0.0, float(v))
    if param_name in {'model.gas_spike_share', 'schedules.wfh_share', 'model.dhw_away_mult', 'model.heating_occupancy_away_mult'}:
        v = max(0.0, min(1.0, float(v)))
    if param_name in {
        'model.heating_peak_morning_mult',
        'model.heating_peak_evening_mult',
        'model.heating_winter_morning_mult',
        'model.heating_winter_evening_mult',
        'model.dhw_peak_morning_mult',
        'model.dhw_peak_evening_mult',
    }:
        v = max(0.6, min(1.8, float(v)))
    if param_name == 'schedules.jitter_hours':
        v = int(max(0, round(float(v))))
    return v


def run_stage(stage_cfg: dict):
    global run_counter
    stage_name = stage_cfg['name']
    stage_windows = STAGE_WINDOWS.get(stage_name, STAGE_WINDOWS['W2_joint_refine'])
    month_filter = stage_cfg.get('month_filter')
    monthly_hdd_min = stage_cfg.get('monthly_hdd_min')
    monthly_hdd_max = stage_cfg.get('monthly_hdd_max')
    weights = stage_cfg['weights']

    print(f"\n=== Stage {stage_name}: {stage_cfg['description']} ===")
    for param_name, candidates in stage_cfg['params']:
        best = None
        for candidate in candidates:
            trial = dict(current)
            candidate_value = _candidate_value(stage_cfg, param_name, current.get(param_name), candidate)
            trial[param_name] = candidate_value

            run_counter += 1
            run_label = f"{stage_name}_r{run_counter:04d}_{param_name.replace('.', '_')}"
            t0 = time.perf_counter()
            seg_ts, cfg_path = run_with_segmentations(
                wf_sample=wf,
                windows=stage_windows,
                override_params=trial,
                configs_dir=CONFIGS_DIR,
                climate_parquet=CLIMATE_PARQUET,
                local_tz=LOCAL_TZ,
                run_label=run_label,
            )
            abm_hourly, abm_monthly = abm_profiles_from_seg_ts(seg_ts, local_tz=LOCAL_TZ)
            marg = score_against_serl_marginals(
                abm_hourly=abm_hourly,
                abm_monthly=abm_monthly,
                hourly_targets=hourly_targets,
                daily_targets=daily_targets,
                target_year=TARGET_YEAR,
                seg3_vars=seg3_vars,
                month_filter=month_filter,
                monthly_hdd_min=monthly_hdd_min,
                monthly_hdd_max=monthly_hdd_max,
            )
            objective_raw = scalar_objective(marg, weights)
            guardrail_penalty = 0.0
            guardrail_df = pd.DataFrame()
            if 'guardrails' in stage_cfg:
                grd = stage_cfg['guardrails']
                guardrail_penalty, guardrail_df = monthly_ratio_guardrail_penalty(
                    abm_monthly=abm_monthly,
                    daily_targets=daily_targets,
                    target_year=TARGET_YEAR,
                    seg3_vars=seg3_vars,
                    months=[int(x) for x in grd.get('months', [6, 7, 8])],
                    ratio_band=(float(grd.get('ratio_band', [0.9, 1.1])[0]), float(grd.get('ratio_band', [0.9, 1.1])[1])),
                    fuel_weights=dict(grd.get('fuel_weights', {'gas': 0.0, 'electric': 0.0})),
                )
            objective = float(objective_raw + guardrail_penalty)

            latest_scores_by_stage[stage_name] = marg.copy()
            elapsed = float(time.perf_counter() - t0)
            trace_rows.append({
                'stage': stage_name,
                'param': param_name,
                'candidate_raw': candidate,
                'candidate_value': candidate_value,
                'objective_raw': objective_raw,
                'guardrail_penalty': guardrail_penalty,
                'objective': objective,
                'elapsed_s': elapsed,
                'config_path': str(cfg_path),
            })

            print(f"{param_name}={candidate_value} (raw {candidate}) -> objective={objective:.4f} raw={objective_raw:.4f} guard={guardrail_penalty:.4f}")
            if best is None or (np.isfinite(objective) and objective < best['objective']):
                best = {
                    'value': candidate_value,
                    'objective': objective,
                    'objective_raw': objective_raw,
                    'guardrail_penalty': guardrail_penalty,
                    'scores': marg,
                    'guardrail_df': guardrail_df,
                    'elapsed_s': elapsed,
                }

        if best is not None:
            current[param_name] = best['value']
            score_means = (best['scores'].groupby('component', as_index=False)
                .agg(mean_value=('value', 'mean')))
            row = {
                'stage': stage_name,
                'param': param_name,
                'selected_value': best['value'],
                'best_objective': best['objective'],
                'best_objective_raw': best.get('objective_raw', best['objective']),
                'best_guardrail_penalty': best.get('guardrail_penalty', 0.0),
                'selected_elapsed_s': best['elapsed_s'],
            }
            for _, rr in score_means.iterrows():
                row[f"comp_{rr['component']}"] = rr['mean_value']
            stage_metrics_rows.append(row)
            latest_scores_by_stage[stage_name] = best['scores'].copy()
            if isinstance(best.get('guardrail_df'), pd.DataFrame) and not best['guardrail_df'].empty:
                grm = best['guardrail_df'].copy()
                grm['stage'] = stage_name
                grm['param'] = param_name
                guardrail_rows.extend(grm.to_dict(orient='records'))
            print(f"CHOSEN {param_name}={best['value']} objective={best['objective']:.4f}")


for stage in STAGE_DEFS:
    run_stage(stage)

trace_df = pd.DataFrame(trace_rows)
stage_metrics_df = pd.DataFrame(stage_metrics_rows)

trace_path = OUT_DIR / 'staged_coordinate_descent_trace.csv'
stage_metrics_path = OUT_DIR / 'per_stage_metrics.csv'
trace_df.to_csv(trace_path, index=False)
stage_metrics_df.to_csv(stage_metrics_path, index=False)

stage_components = []
for stage_name, df in latest_scores_by_stage.items():
    if df is None or df.empty:
        continue
    tmp = df.copy()
    tmp['stage'] = stage_name
    stage_components.append(tmp)

stage_components_df = pd.concat(stage_components, ignore_index=True) if stage_components else pd.DataFrame(columns=['stage','seg3_var','component','value'])
stage_components_path = OUT_DIR / 'stage_objective_components_by_seg3_var.csv'
stage_components_df.to_csv(stage_components_path, index=False)
guardrail_df = pd.DataFrame(guardrail_rows)
guardrail_path = OUT_DIR / 'stage_guardrail_penalties.csv'
guardrail_df.to_csv(guardrail_path, index=False)

final_params_path = OUT_DIR / 'final_params.yaml'
with final_params_path.open('w', encoding='utf-8') as fh:
    yaml.safe_dump(
        {
            'params': current,
            'seg3_vars': seg3_vars,
            'target_year': int(TARGET_YEAR),
            'stage_definitions': [
                {
                    'name': s['name'],
                    'description': s['description'],
                    'month_filter': s.get('month_filter'),
                    'monthly_hdd_min': s.get('monthly_hdd_min'),
                    'monthly_hdd_max': s.get('monthly_hdd_max'),
                }
                for s in STAGE_DEFS
            ],
        },
        fh,
        sort_keys=False,
    )

print('Trace:', trace_path)
print('Stage metrics:', stage_metrics_path)
print('Stage components:', stage_components_path)
print('Guardrails:', guardrail_path)
print('Final params:', final_params_path)
display(trace_df.sort_values('objective').head(20))

In [ ]:
# ---- Quick diagnostics ----
if trace_df.empty:
    print('No calibration trace rows.')
else:
    by_stage_param = (trace_df.groupby(['stage', 'param', 'candidate_value'], as_index=False)
        .agg(best_objective=('objective', 'min'), mean_elapsed_s=('elapsed_s', 'mean')))
    display(by_stage_param.sort_values(['stage', 'param', 'best_objective']))

    fig, ax = plt.subplots(figsize=(11, 4))
    for (stage_name, pname), g in by_stage_param.groupby(['stage', 'param'], observed=True):
        g = g.sort_values('candidate_value')
        ax.plot(g['candidate_value'].astype(str), g['best_objective'], marker='o', label=f"{stage_name}: {pname}")
    ax.set_title('Objective by candidate and stage')
    ax.set_ylabel('objective (lower is better)')
    ax.set_xlabel('candidate value')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    fig.tight_layout()
    plt.show()

if not stage_metrics_df.empty:
    display(stage_metrics_df)

## Fit Visualizations (ABM vs SERL)

This section re-runs the calibrated parameter set and exports comparison artefacts:
- per-`seg3_var` component scores
- per-`seg3_var` trend overlays (hourly electricity/gas, monthly electricity/gas)


In [ ]:
# ---- Final evaluation run using calibrated params ----
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Use full-year for trend plots by default; set False for quick turnaround.
PLOT_FULL_YEAR = True

if PLOT_FULL_YEAR:
    eval_windows = [
        Window('full_year', pd.Timestamp(f'{TARGET_YEAR}-01-01T00:00:00Z'), pd.Timestamp(f'{TARGET_YEAR+1}-01-01T00:00:00Z')),
    ]
else:
    eval_windows = STAGE_WINDOWS['W2_joint_refine']

final_seg_ts, final_cfg_path = run_with_segmentations(
    wf_sample=wf,
    windows=eval_windows,
    override_params=current,
    configs_dir=CONFIGS_DIR,
    climate_parquet=CLIMATE_PARQUET,
    local_tz=LOCAL_TZ,
    run_label='final_evaluation',
)

final_hourly, final_monthly = abm_profiles_from_seg_ts(final_seg_ts, local_tz=LOCAL_TZ)
final_scores = score_against_serl_marginals(
    abm_hourly=final_hourly,
    abm_monthly=final_monthly,
    hourly_targets=hourly_targets,
    daily_targets=daily_targets,
    target_year=TARGET_YEAR,
    seg3_vars=seg3_vars,
)

final_scores_path = OUT_DIR / 'final_component_scores.csv'
final_hourly_path = OUT_DIR / 'final_abm_hourly_by_seg.csv'
final_monthly_path = OUT_DIR / 'final_abm_monthly_by_seg.csv'
final_seg_ts_path = OUT_DIR / 'final_segmentation_timeseries.csv'

final_scores.to_csv(final_scores_path, index=False)
final_hourly.to_csv(final_hourly_path, index=False)
final_monthly.to_csv(final_monthly_path, index=False)
final_seg_ts.to_csv(final_seg_ts_path, index=False)

objective_components_by_seg3 = (final_scores.pivot_table(index='seg3_var', columns='component', values='value', aggfunc='mean')
    .reset_index())
objective_components_by_seg3_path = OUT_DIR / 'objective_components_by_seg3_var.csv'
objective_components_by_seg3.to_csv(objective_components_by_seg3_path, index=False)


winter_uplift_slopes = regress_winter_uplift_vs_hdd(
    abm_monthly=final_monthly,
    daily_targets=daily_targets,
    target_year=TARGET_YEAR,
    seg3_vars=seg3_vars,
)
winter_uplift_slopes_path = OUT_DIR / 'winter_uplift_hdd_slopes_by_seg3.csv'
winter_uplift_slopes.to_csv(winter_uplift_slopes_path, index=False)

print('Final config used:', final_cfg_path)
print('Saved:', final_scores_path)
print('Saved:', final_hourly_path)
print('Saved:', final_monthly_path)
print('Saved:', final_seg_ts_path)
print('Saved:', objective_components_by_seg3_path)
print('Saved:', winter_uplift_slopes_path)
print('Final weighted objective:', scalar_objective(final_scores, DEFAULT_WEIGHTS))

display(final_scores.groupby('component', as_index=False).agg(mean=('value','mean')).sort_values('mean'))

In [ ]:
# ---- Score heatmap by seg3_var x component ----
if final_scores.empty:
    print('No scores to plot.')
else:
    pivot = final_scores.pivot_table(index='seg3_var', columns='component', values='value', aggfunc='mean')
    pivot = pivot.sort_index().reindex(sorted(pivot.columns), axis=1)

    fig, ax = plt.subplots(figsize=(10, 4 + 0.5*len(pivot.index)))
    im = ax.imshow(pivot.to_numpy(dtype=float), aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title('ABM vs SERL objective components by seg3_var (lower is better)')
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('component value')
    fig.tight_layout()

    hm_path = FIG_DIR / 'score_heatmap_by_seg3_component.png'
    fig.savefig(hm_path, dpi=170)
    plt.show()
    print('Saved:', hm_path)

In [ ]:
# ---- ABM vs SERL overlays by seg3_var/fuel ----

def _serl_hourly_slice(seg3_var: str, quantity: str) -> pd.DataFrame:
    return hourly_targets[(hourly_targets['year'].astype(int) == int(TARGET_YEAR))
        & (hourly_targets['weekday_weekend'].astype(str) == 'both')
        & (hourly_targets['heating_fuel'].astype(str) == 'All')
        & (hourly_targets['has_pv'].astype(str).isin(['No', 'All']))
        & (hourly_targets['seg3_var'].astype(str) == seg3_var)
        & (hourly_targets['quantity'].astype(str) == quantity)].copy()


def _serl_monthly_slice(seg3_var: str, quantity: str) -> pd.DataFrame:
    return daily_targets[(daily_targets['period_type'].astype(str) == 'monthly')
        & (daily_targets['year'].astype(int) == int(TARGET_YEAR))
        & (daily_targets['weekday_weekend'].astype(str) == 'both')
        & (daily_targets['heating_fuel'].astype(str) == 'All')
        & (daily_targets['has_pv'].astype(str).isin(['No', 'All']))
        & (daily_targets['seg3_var'].astype(str) == seg3_var)
        & (daily_targets['quantity'].astype(str) == quantity)].copy()


def _top_values(seg3_var: str, top_k: int = 3) -> list[str]:
    src = _serl_hourly_slice(seg3_var, 'Electricity imports')
    if src.empty:
        return []
    top = (src.groupby('seg3_value', observed=True)['n_rounded']
        .max().sort_values(ascending=False).head(top_k).index.tolist())
    return [str(v) for v in top]


def _plot_overlay(seg3_var: str, value: str, fuel: str):
    q = 'Electricity imports' if fuel == 'electric' else 'Gas'
    abm_col_h = 'elec_kwh' if fuel == 'electric' else 'gas_kwh'
    abm_col_m = 'elec_kwh_per_home_day' if fuel == 'electric' else 'gas_kwh_per_home_day'

    ah = final_hourly[(final_hourly['segmentation'] == seg3_var) & (final_hourly['value'].astype(str) == str(value))].copy()
    am = final_monthly[(final_monthly['segmentation'] == seg3_var) & (final_monthly['value'].astype(str) == str(value))].copy()
    sh = _serl_hourly_slice(seg3_var, q)
    sm = _serl_monthly_slice(seg3_var, q)
    sh = sh[sh['seg3_value'].astype(str) == str(value)].copy()
    sm = sm[sm['seg3_value'].astype(str) == str(value)].copy()

    if ah.empty or (sh.empty and sm.empty):
        return None

    fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))

    ax = axs[0]
    ax.plot(ah['local_hour'], ah[abm_col_h], marker='o', label='ABM')
    if not sh.empty:
        s = sh.groupby('hour', as_index=False).agg(v=('mean_kwh','mean')).sort_values('hour')
        ax.plot(s['hour'], s['v'], marker='o', label='SERL')
    ax.set_title(f'Hourly {fuel} (kWh/home/hour)')
    ax.set_xlabel('Hour')
    ax.set_ylabel('kWh/home/hour')
    ax.set_xticks(range(0,24,2))
    ax.legend()

    ax = axs[1]
    if not am.empty:
        ax.plot(am['month'], am[abm_col_m], marker='o', label='ABM')
    if not sm.empty:
        s = sm.groupby('month', as_index=False).agg(v=('mean','mean')).sort_values('month')
        ax.plot(s['month'], s['v'], marker='o', label='SERL')
    ax.set_title(f'Monthly {fuel} (kWh/home/day)')
    ax.set_xlabel('Month')
    ax.set_ylabel('kWh/home/day')
    ax.legend()

    fig.suptitle(f'{seg3_var} = {value} ({fuel})')
    fig.tight_layout()
    return fig


plot_index = []
for seg3 in seg3_vars:
    vals = _top_values(seg3, top_k=3)
    for val in vals:
        for fuel in ['electric', 'gas']:
            fig = _plot_overlay(seg3, val, fuel)
            if fig is None:
                continue
            safe_val = str(val).replace(' ', '_').replace('/', '_')
            fname = f"overlay_{seg3}_{safe_val}_{fuel}.png"
            out = FIG_DIR / fname
            fig.savefig(out, dpi=170)
            plt.close(fig)
            plot_index.append({'seg3_var': seg3, 'value': val, 'fuel': fuel, 'path': str(out)})

plot_index_df = pd.DataFrame(plot_index)
plot_index_path = OUT_DIR / 'overlay_plot_index.csv'
plot_index_df.to_csv(plot_index_path, index=False)
print('Saved plot index:', plot_index_path)
display(plot_index_df.head(30))